# CGR Transcriptomic Benchmark (LINCS L1000 + Postmortem Brain DEGs + Ketamine Mimicry)

Self-contained Google Colab pipeline querying **SigCom LINCS** live to pull L1000 perturbation signatures for the CGR stack and ketamine, score module-restricted disease reversal against postmortem brain DEGs (**MDD, BIP, SCZ, OCD**), and compute pairwise **ketamine-mimicry** and **component synergy/antagonism**.

**Fixes & upgrades over the draft script:**
1. **Embedded 121-gene `cgr_anchors.csv` + fixed module matching:** Correctly parses semicolon-separated modules (`LTP_init`, `TrkB_mTOR`, `Actin_spine`, plus optional `Adhesion_PSD` / Pharm anchors) instead of failing on exact-string `.isin(["01 LTP", ...])`.
2. **Verified SigCom LINCS retrieval:** Downloads each signature's S3-hosted tab-separated file via `meta.persistent_id` from `signatures/find` (with fallback to Data API `/fetch`), prioritizing neural cell lines (`NEU`, `NPC`, `SHSY5Y`, `SKNSH`) and collapsing replicates via robust median consensus.
3. **Graceful DEG handling:** Any missing postmortem DEG CSV (`deg_MDD.csv`, `deg_BIP.csv`, `deg_SCZ.csv`, `deg_OCD.csv`, `deg_ketamine_ext.csv`) is skipped with a clear log rather than crashing, so the **L1000 Ketamine-Mimicry & Antagonism Matrix** always runs immediately.
4. **Automatic `.zip` export** via `google.colab.files.download()`.


In [1]:
# @title 1. Configuration, Embedded 121-Gene Anchors, & Optional DEG Folder/Upload
import os, io, time, json, zipfile
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

MOUNT_DRIVE = True              # @param {type:"boolean"}
DRIVE_DEG_FOLDER = ""           # @param {type:"string"}
PREFER_NEURAL_CELLS = True      # @param {type:"boolean"}
INCLUDE_ADHESION_PSD = False    # @param {type:"boolean"}
CYP2D6_INHIBITOR = "bupropion"  # @param ["bupropion", "paroxetine", "duloxetine", "quinidine"]
MAX_SIGS_PER_COMPOUND = 40      # @param {type:"integer"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ANCHORS_CSV = """symbol,ensembl_id,module,is_pharm_anchor,functional_family,pathway_sets
GRIN1,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0035235;GO:0007215
GRIN2A,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0060291;GO:0035235;GO:0007215
GRIN2B,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0060291;GO:0035235;GO:0007215
GRIN2C,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0060291;GO:0035235;GO:0007215
GRIN2D,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0060291;GO:0035235;GO:0007215
GRIA1,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724
GRIA2,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0035235;GO:0007215
GRIA3,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04724;GO:0007215
GRIA4,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04724;GO:0007215
GRM2,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04724;GO:0007215
GRM3,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04724;GO:0007215
SLC1A1,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC1A2,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC1A3,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC1A6,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC1A7,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
GLUL,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC38A1,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC38A2,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC38A3,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
GLS,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
GLS2,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC17A6,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC17A7,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC17A8,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
VAMP2,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,GO:0060291
STX4,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,GO:0060291
CACNA1C,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
CALM1,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
CALM2,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
CALM3,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
CAMK2A,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
CAMK2B,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;GO:0061001
ADCY1,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
ADCY8,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
PRKACA,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
PRKACB,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
PRKACG,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
RAPGEF3,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720
RAP1A,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
RAP1B,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
HRAS,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04151;hsa04810
BRAF,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04810
RAF1,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04151;hsa04810
MAP2K1,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04151;hsa04810
MAP2K2,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04151;hsa04810
MAPK1,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724;hsa04722;hsa04150;hsa04151;hsa04810
MAPK3,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724;hsa04722;hsa04150;hsa04151;hsa04810
RPS6KA1,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150
RPS6KA2,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150
RPS6KA3,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150
RPS6KA6,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150
CREBBP,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720
EP300,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720
ATF4,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04151
BDNF,,TrkB_mTOR;Adhesion_PSD,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04151;GO:0007416
NTRK2,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04151
PIK3CA,,TrkB_mTOR;Actin_spine,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04150;hsa04151;hsa04810
AKT1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04150;hsa04151
AKT2,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04150;hsa04151
AKT3,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04150;hsa04151
STK11,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
PRKAA1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
PRKAA2,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
TSC1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
TSC2,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
RHEB,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
MTOR,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151;GO:0031929
RPTOR,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151;GO:0031929
RPS6KB1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151;GO:0031929
EIF4E,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
EIF4EBP1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151;GO:0031929
RAC1,,TrkB_mTOR;Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04722;hsa04151;hsa04810;GO:0007015
CDC42,,TrkB_mTOR;Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04722;hsa04810;GO:0007015;GO:0007266
PAK1,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
PAK3,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810;GO:0060997
CFL1,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810;GO:0061001;GO:0007266
ACTB,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
ACTG1,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
ACTR2,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
ACTR3,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
WASL,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810;GO:0060997;GO:0007015
DBN1,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",GO:0061001;GO:0007015
PLA2G4A,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4B,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4C,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4D,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4E,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4F,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLD1,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLD2,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PIP5K1A,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
PIP5K1B,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
PIP5K1C,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
DLG4,,LTP_init;TrkB_mTOR;Actin_spine;Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04724;GO:0060997
SHANK1,,LTP_init;Actin_spine,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04724;GO:0060997
SHANK2,,LTP_init;Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04724;GO:0060291;GO:0007416
SHANK3,,LTP_init;TrkB_mTOR;Actin_spine;Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04724;GO:0060997;GO:0061001;GO:0007416
NLGN1,,Actin_spine;Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",GO:0061001;hsa04514;GO:0007416;GO:0097104
NRXN1,,Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04514;GO:0007416;GO:0097104
CDH2,,Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04514;GO:0007416
BCL2,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04722;hsa04151
BCL2L1,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04151
BAD,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04722;hsa04151
TP53,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04722;hsa04151
CDKN1A,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04151
CYP2D6,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SLC7A11,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PDE4A,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PDE4B,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PDE4C,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PDE4D,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SLC6A4,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SIGMAR1,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SIRT1,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PPARGC1A,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
NAMPT,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PCYT1A,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
CHKA,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SYP,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SNAP25,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),"""
Path("data/cgr_anchors.csv").write_text(ANCHORS_CSV + "\n", encoding="utf-8")
anchors = pd.read_csv("data/cgr_anchors.csv")

# Match module tokens cleanly (supports both LTP_init/TrkB_mTOR/Actin_spine and 01 LTP/02 TrkB-mTOR/03 Spines)
target_tokens = {"LTP_INIT", "01 LTP", "TRKB_MTOR", "02 TRKB-MTOR", "02 TRKB–MTOR", "ACTIN_SPINE", "03 SPINES"}
if INCLUDE_ADHESION_PSD:
    target_tokens.update({"ADHESION_PSD", "04 ADHESION"})

def matches_target_module(mod_str):
    if pd.isna(mod_str):
        return False
    parts = [p.strip().upper() for p in str(mod_str).replace(",", ";").split(";")]
    return any(p in target_tokens for p in parts)

module_genes = sorted(anchors[anchors["module"].apply(matches_target_module)]["symbol"].dropna().unique().tolist())
all_anchor_genes = sorted(anchors["symbol"].dropna().unique().tolist())
print(f"[OK] Embedded {len(all_anchor_genes)} total anchors; {len(module_genes)} genes in target synaptic modules.")

# Link any existing DEG CSVs from Drive folder if provided
DEG_NAMES = ["deg_MDD.csv", "deg_BIP.csv", "deg_SCZ.csv", "deg_OCD.csv", "deg_ketamine_ext.csv"]
if DRIVE_DEG_FOLDER.strip():
    droot = Path(DRIVE_DEG_FOLDER.strip())
    for fname in DEG_NAMES:
        src = droot / fname
        dst = Path("data") / fname
        if src.exists() and not dst.exists():
            os.symlink(src, dst)
            print(f"[OK] Linked {src} -> {dst}")

present = [f for f in DEG_NAMES if (Path("data") / f).exists()]
missing = [f for f in DEG_NAMES if not (Path("data") / f).exists()]
print(f"[INFO] DEG files ready: {present}")
if missing:
    print(f"[INFO] Optional DEG files not yet in data/: {missing} (upload in Cell 2 if available, or run L1000 Ketamine/Antagonism benchmark immediately).")


Mounted at /content/drive
[OK] Embedded 121 total anchors; 104 genes in target synaptic modules.
[INFO] DEG files ready: []
[INFO] Optional DEG files not yet in data/: ['deg_MDD.csv', 'deg_BIP.csv', 'deg_SCZ.csv', 'deg_OCD.csv', 'deg_ketamine_ext.csv'] (upload in Cell 2 if available, or run L1000 Ketamine/Antagonism benchmark immediately).


In [2]:
# @title 2. (Optional) Upload Postmortem Brain DEG CSVs (deg_MDD.csv, deg_BIP.csv, deg_SCZ.csv, deg_OCD.csv, deg_ketamine_ext.csv)
UPLOAD_DEG_FILES = False  # @param {type:"boolean"}

if UPLOAD_DEG_FILES:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        Path(fname).rename(Path("data") / Path(fname).name)
        print(f"[OK] Saved data/{Path(fname).name}")
else:
    print("[SKIP] Set UPLOAD_DEG_FILES = True above if you want to upload DEG CSVs now.")


[SKIP] Set UPLOAD_DEG_FILES = True above if you want to upload DEG CSVs now.


In [3]:
# @title 2b. Auto-Download Public Postmortem Brain DEG Tables (SCZ, BIP, MDD, OCD)
import re, io, requests
import pandas as pd
from pathlib import Path

Path("data").mkdir(exist_ok=True)
!pip -q install openpyxl

UA = {"User-Agent": "Mozilla/5.0 (CGR-Transcriptomic-Benchmark/1.0)"}

def map_ensembl_to_symbol(ensembl_ids):
    """Map Ensembl gene IDs to HGNC symbols via MyGene.info batch API."""
    clean_ids = list({str(e).split(".")[0] for e in ensembl_ids if pd.notna(e)})
    mapping = {}
    for i in range(0, len(clean_ids), 1000):
        chunk = clean_ids[i:i+1000]
        r = requests.post(
            "https://mygene.info/v3/gene",
            json={"ids": chunk, "fields": "symbol"},
            headers=UA,
            timeout=60,
        )
        if r.ok:
            for item in r.json():
                if isinstance(item, dict) and "symbol" in item and "query" in item:
                    mapping[item["query"]] = item["symbol"]
    return mapping

# -------------------------------------------------------------------------
# 1) SCZ & BIP from PsychENCODE DER-13 (resource.psychencode.org)
# -------------------------------------------------------------------------
try:
    der13_url = "http://resource.psychencode.org/Datasets/Derived/DEXgenes_CoExp/DER-13_Disorder_DEX_Genes.csv"
    r = requests.get(der13_url, headers=UA, timeout=60)
    r.raise_for_status()
    der = pd.read_csv(io.StringIO(r.text))
    cols = {c.lower(): c for c in der.columns}

    # Resolve gene symbol column (map from Ensembl ID if needed)
    sym_col = next((cols[k] for k in ["gene_name", "symbol", "hgnc_symbol", "gene_symbol"] if k in cols), None)
    if sym_col is None:
        ens_col = next((cols[k] for k in ["ensembl_gene_id", "ensembl_id", "gene_id", "id"] if k in cols), der.columns[0])
        ens_clean = der[ens_col].astype(str).str.split(".").str[0]
        ens_map = map_ensembl_to_symbol(ens_clean)
        der["symbol"] = ens_clean.map(ens_map)
    else:
        der["symbol"] = der[sym_col]

    for tag, out_name in [("SCZ", "deg_SCZ.csv"), ("BD", "deg_BIP.csv")]:
        fc_col = next((c for c in der.columns if tag.lower() in c.lower() and "log2fc" in c.lower()), None)
        if fc_col is None:
            fc_col = next((c for c in der.columns if tag.lower() in c.lower() and "fc" in c.lower()), None)
        if fc_col:
            sub = der[["symbol", fc_col]].rename(columns={fc_col: "log2FC"}).dropna()
            sub.to_csv(f"data/{out_name}", index=False)
            print(f"[OK] Saved data/{out_name} ({len(sub):,} genes) from PsychENCODE DER-13")
except Exception as e:
    print(f"[WARN] PsychENCODE DER-13 download failed: {e}")

# -------------------------------------------------------------------------
# 2) MDD (and fallback SCZ/BIP) from Gandal et al. (2018) GitHub repo
# -------------------------------------------------------------------------
try:
    repo_api = "https://api.github.com/repos/mgandal/Shared-molecular-neuropathology-across-major-psychiatric-disorders-parallels-polygenic-overlap/git/trees/master?recursive=1"
    tree = requests.get(repo_api, headers=UA, timeout=45).json().get("tree", [])
    csv_paths = [
        t["path"] for t in tree
        if t["path"].lower().endswith(".csv")
        and any(k in t["path"].lower() for k in ["table", "deg", "microarray", "meta", "summary", "diff"])
    ]
    mdd_saved = False
    for path in csv_paths:
        raw_url = f"https://raw.githubusercontent.com/mgandal/Shared-molecular-neuropathology-across-major-psychiatric-disorders-parallels-polygenic-overlap/master/{path}"
        resp = requests.get(raw_url, headers=UA, timeout=45)
        if not resp.ok:
            continue
        try:
            df = pd.read_csv(io.StringIO(resp.text))
        except Exception:
            continue
        mdd_fc = next((c for c in df.columns if "mdd" in c.lower() and any(k in c.lower() for k in ["beta", "log2fc", "logfc", "fc"])), None)
        if not mdd_fc:
            continue

        # Identify gene symbol or Ensembl column
        sym_c = next((c for c in df.columns if c.lower() in {"hgnc_symbol", "symbol", "gene_symbol", "gene_name", "external_gene_name"}), None)
        if sym_c is None:
            ens_c = next((c for c in df.columns if "ensembl" in c.lower() or c.lower() in {"x", "unnamed: 0", "gene"}), df.columns[0])
            vals = df[ens_c].astype(str)
            if vals.str.startswith("ENSG").mean() > 0.5:
                ens_map = map_ensembl_to_symbol(vals.str.split(".").str[0])
                df["symbol"] = vals.str.split(".").str[0].map(ens_map)
            else:
                df["symbol"] = vals
        else:
            df["symbol"] = df[sym_c]

        sub = df[["symbol", mdd_fc]].rename(columns={mdd_fc: "log2FC"}).dropna()
        if len(sub) > 100:
            sub.to_csv("data/deg_MDD.csv", index=False)
            print(f"[OK] Saved data/deg_MDD.csv ({len(sub):,} genes) from Gandal et al. ({path})")
            mdd_saved = True
            break
    if not mdd_saved:
        print("[WARN] Could not auto-locate MDD summary CSV in Gandal GitHub repo.")
except Exception as e:
    print(f"[WARN] Gandal et al. MDD download failed: {e}")

# -------------------------------------------------------------------------
# 3) OCD from Piantadosi et al. (2021) Translational Psychiatry (Open Access)
# -------------------------------------------------------------------------
try:
    art_url = "https://www.nature.com/articles/s41398-021-01290-1"
    html = requests.get(art_url, headers=UA, timeout=45).text
    xlsx_links = sorted(set(re.findall(r"https://static-content\.springer\.com/esm/[^\"']+\.xlsx?", html)))
    ocd_saved = False
    for url in xlsx_links:
        r = requests.get(url, headers=UA, timeout=60)
        if not r.ok:
            continue
        xls = pd.ExcelFile(io.BytesIO(r.content))
        for sheet in xls.sheet_names:
            df = xls.parse(sheet)
            # Check if header is shifted down by 1-2 rows
            for skip in [0, 1, 2]:
                if skip > 0:
                    df = xls.parse(sheet, skiprows=skip)
                cols_l = {str(c).lower().strip(): c for c in df.columns}
                sym_c = next((cols_l[k] for k in ["symbol", "gene_symbol", "gene", "gene_name", "hgnc_symbol", "external_gene_name"] if k in cols_l), None)
                fc_c = next((c for c in df.columns if any(k in str(c).lower() for k in ["log2fc", "logfc", "log2foldchange", "foldchange"])), None)
                if sym_c and fc_c:
                    sub = df[[sym_c, fc_c]].rename(columns={sym_c: "symbol", fc_c: "log2FC"}).dropna()
                    if len(sub) >= 100:
                        sub.to_csv("data/deg_OCD.csv", index=False)
                        print(f"[OK] Saved data/deg_OCD.csv ({len(sub):,} genes) from Piantadosi et al. ({sheet})")
                        ocd_saved = True
                        break
            if ocd_saved:
                break
        if ocd_saved:
            break
    if not ocd_saved:
        print("[WARN] Could not auto-parse OCD DEG sheet from Piantadosi et al. (2021).")
except Exception as e:
    print(f"[WARN] Piantadosi et al. OCD download failed: {e}")

print("\nReady DEG files in data/:", sorted(p.name for p in Path("data").glob("deg_*.csv")))

[OK] Saved data/deg_MDD.csv (19,064 genes) from Gandal et al. (results/tables/Manuscript/TableS1 - Microarray Meta-Analyses.csv)
[OK] Saved data/deg_OCD.csv (541 genes) from Piantadosi et al. (Caudate)

Ready DEG files in data/: ['deg_MDD.csv', 'deg_OCD.csv']


In [4]:
# @title 2c. Extract SCZ & BIP from Gandal et al. TableS1
import io, requests
import pandas as pd
from pathlib import Path

url = "https://raw.githubusercontent.com/mgandal/Shared-molecular-neuropathology-across-major-psychiatric-disorders-parallels-polygenic-overlap/master/results/tables/Manuscript/TableS1%20-%20Microarray%20Meta-Analyses.csv"
r = requests.get(url, timeout=45)
r.raise_for_status()
df = pd.read_csv(io.StringIO(r.text))

# Identify gene symbol column
sym_c = next((c for c in df.columns if c.lower() in {"hgnc_symbol", "symbol", "gene_symbol", "gene_name", "external_gene_name"}), None)
if sym_c is None:
    ens_c = next((c for c in df.columns if "ensembl" in c.lower() or c.lower() in {"x", "unnamed: 0", "gene"}), df.columns[0])
    vals = df[ens_c].astype(str)
    if vals.str.startswith("ENSG").mean() > 0.5:
        ens_map = map_ensembl_to_symbol(vals.str.split(".").str[0])
        df["symbol"] = vals.str.split(".").str[0].map(ens_map)
    else:
        df["symbol"] = vals
else:
    df["symbol"] = df[sym_c]

for keywords, out_name in [(["scz"], "deg_SCZ.csv"), (["bd", "bip"], "deg_BIP.csv")]:
    fc_col = next(
        (c for c in df.columns
         if any(k in c.lower() for k in keywords)
         and any(m in c.lower() for m in ["beta", "log2fc", "logfc", "fc"])),
        None,
    )
    if fc_col:
        sub = df[["symbol", fc_col]].rename(columns={fc_col: "log2FC"}).dropna()
        sub.to_csv(f"data/{out_name}", index=False)
        print(f"[OK] Saved data/{out_name} ({len(sub):,} genes) using column '{fc_col}'")
    else:
        print(f"[WARN] Could not find column for {out_name}. Available columns: {list(df.columns)}")

print("\nReady DEG files in data/:", sorted(p.name for p in Path("data").glob("deg_*.csv")))

[OK] Saved data/deg_SCZ.csv (12,115 genes) using column 'SCZ.beta_log2FC'
[OK] Saved data/deg_BIP.csv (12,110 genes) using column 'BD.beta_log2FC'

Ready DEG files in data/: ['deg_BIP.csv', 'deg_MDD.csv', 'deg_OCD.csv', 'deg_SCZ.csv']


In [8]:
# @title 3. Fetch LINCS L1000 Perturbation Signatures (Deterministic + Matched Cell-Line Control)
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Toggle False for apples-to-apples all_cell_lines control across Core Stack & CYP2D6i swaps
PREFER_NEURAL_CELLS = False  # @param {type:"boolean"}
MAX_SIGS_PER_COMPOUND = 60   # @param {type:"integer"}
ENABLE_ILIKE_FALLBACK = False  # @param {type:"boolean"}  # False avoids slow unindexed scans that trigger 503/timeouts

SIGCOM_META = "https://maayanlab.cloud/sigcom-lincs/metadata-api"
SIGCOM_DATA = "https://maayanlab.cloud/sigcom-lincs/data-api/api/v1"

# Aligned synonyms (matches Cell 6 CYP2D6i aliases to eliminate cross-cell drift)
CYP2D6_ALIAS_MAP = {
    "bupropion":  ["bupropion", "bupropion hydrochloride", "amfebutamone"],
    "paroxetine": ["paroxetine", "paroxetine hydrochloride", "paroxetine hemihydrate"],
    "duloxetine": ["duloxetine", "duloxetine hydrochloride"],
    "quinidine":  ["quinidine", "quinidine sulfate", "quinidine gluconate", "quinidine hydrochloride"],
    "fluoxetine": ["fluoxetine", "fluoxetine hydrochloride"],
}

COMPOUNDS = {
    "DXM": {
        "names": ["dextrorphan"],
        "pert_ids": [],
    },
    "Piracetam": {
        "names": ["piracetam"],
        "pert_ids": [],
    },
    "CYP2D6i": {
        "names": CYP2D6_ALIAS_MAP.get(CYP2D6_INHIBITOR, [CYP2D6_INHIBITOR, f"{CYP2D6_INHIBITOR} hydrochloride"]),
        "pert_ids": ["BRD-A31159102"] if CYP2D6_INHIBITOR == "fluoxetine" else [],
    },
    "Forskolin": {
        "names": ["forskolin", "colforsin"],
        "pert_ids": [],
    },
    "Fisetin": {
        "names": ["fisetin"],
        "pert_ids": [],
    },
    "NAC": {
        "names": ["acetylcysteine", "N-acetylcysteine", "N-acetyl-L-cysteine", "L-acetylcysteine", "nac"],
        "pert_ids": ["BRD-K60297835", "BRD-K94390040", "BRD-K05151076"],
    },
    "DHA": {
        "names": ["docosahexaenoic acid", "cis-4,7,10,13,16,19-docosahexaenoic acid", "cervonic acid", "all-cis-docosa-4,7,10,13,16,19-hexaenoic acid"],
        "pert_ids": ["BRD-K03371390"],
    },
    "Citicoline": {
        "names": ["citicoline", "citicoline sodium", "CDP-choline", "cytidine 5'-diphosphocholine", "cytidine diphosphate choline"],
        "pert_ids": [],
    },
    "Glutamine": {
        "names": ["glutamine", "L-glutamine"],
        "pert_ids": [],
    },
    "Ketamine": {
        "names": ["ketamine", "ketamine hydrochloride", "esketamine", "(S)-ketamine"],
        "pert_ids": [],
    },
}

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "CGR-Transcriptomic-Benchmark/1.3"})
retry_strategy = Retry(
    total=5,
    backoff_factor=4,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"],
)
adapter = HTTPAdapter(max_retries=retry_strategy)
SESSION.mount("https://", adapter)
SESSION.mount("http://", adapter)

NEURAL_CELLS = {"NEU", "NPC", "SHSY5Y", "SKNSH", "SH-SY5Y", "SK-N-SH", "GI1", "LN229", "U251MG"}
GENETIC_PERT_TYPES = {"CRISPR", "SHRNA", "OVEREXPRESSION", "TRT_SH", "TRT_OE", "TRT_XPR", "TRT_LIG"}

def parse_persistent_tsv(url):
    resp = SESSION.get(url, timeout=45)
    resp.raise_for_status()
    lines = resp.text.splitlines()
    if len(lines) < 10:
        return pd.Series(dtype=float)
    first_parts = lines[0].split("\t")
    has_header = False
    if len(first_parts) >= 2:
        try:
            float(first_parts[1])
        except ValueError:
            has_header = True
    df = pd.read_csv(
        io.StringIO(resp.text),
        sep="\t",
        header=0 if has_header else None,
        usecols=[0, 1],
        names=["symbol", "val"] if not has_header else None,
    )
    df.columns = ["symbol", "val"]
    df["symbol"] = df["symbol"].astype(str).str.strip().str.upper()
    df["val"] = pd.to_numeric(df["val"], errors="coerce")
    df = df.dropna(subset=["symbol", "val"]).groupby("symbol")["val"].mean()
    return df

def _sort_sigs_deterministic(sigs):
    """Sort signatures deterministically client-side by signature ID (and persistent_id fallback)."""
    return sorted(
        sigs,
        key=lambda s: (
            str(s.get("id", "")),
            str((s.get("meta") or {}).get("persistent_id", "")),
        ),
    )

def query_sigcom_signatures(spec, limit=250):
    pert_names = spec["names"] if isinstance(spec, dict) else ([spec] if isinstance(spec, str) else list(spec))
    pert_ids = spec.get("pert_ids", []) if isinstance(spec, dict) else []

    # 1. Fast indexed lookup by exact case variants of meta.pert_name
    variants = []
    for name in pert_names:
        variants.extend([name, name.lower(), name.upper(), name.title()])
    variants = list(dict.fromkeys(variants))

    fast_query = {
        "filter": {
            "where": {"meta.pert_name": {"inq": variants}},
            "limit": limit,
        }
    }
    r = SESSION.post(f"{SIGCOM_META}/signatures/find", json=fast_query, timeout=90)
    r.raise_for_status()
    sigs = r.json()
    if isinstance(sigs, list) and sigs:
        return _sort_sigs_deterministic(sigs)

    # 2. Fast indexed lookup by Broad pert_id (BRD-...) if provided
    if pert_ids:
        id_query = {
            "filter": {
                "where": {"meta.pert_id": {"inq": pert_ids}},
                "limit": limit,
            }
        }
        r = SESSION.post(f"{SIGCOM_META}/signatures/find", json=id_query, timeout=90)
        r.raise_for_status()
        sigs = r.json()
        if isinstance(sigs, list) and sigs:
            return _sort_sigs_deterministic(sigs)

    # 3. Optional substring ilike fallback (disabled by default to prevent SigCom 503/timeouts)
    if ENABLE_ILIKE_FALLBACK:
        try:
            ilike_query = {
                "filter": {
                    "where": {"meta.pert_name": {"ilike": f"%{pert_names[0]}%"}},
                    "limit": limit,
                }
            }
            r = SESSION.post(f"{SIGCOM_META}/signatures/find", json=ilike_query, timeout=90)
            r.raise_for_status()
            sigs = r.json()
            if isinstance(sigs, list) and sigs:
                return _sort_sigs_deterministic(sigs)
        except requests.exceptions.RequestException:
            pass

    return []

def fetch_l1000_consensus(spec, prefer_neural=None, max_sigs=None, limit=250):
    if prefer_neural is None:
        prefer_neural = PREFER_NEURAL_CELLS
    if max_sigs is None:
        max_sigs = MAX_SIGS_PER_COMPOUND

    primary_name = spec["names"][0] if isinstance(spec, dict) else str(spec)
    try:
        sigs = query_sigcom_signatures(spec, limit=limit)
    except requests.exceptions.RequestException as e:
        print(f"    [ERR] Query failed for '{primary_name}': {e}")
        return pd.Series(dtype=float), {"n_found": 0, "n_used": 0, "cell_mode": "timeout_or_error"}

    # Client-side filter: exclude genetic perturbations & keep deterministic order
    sigs = _sort_sigs_deterministic([
        s for s in sigs
        if str((s.get("meta") or {}).get("pert_type", "")).upper() not in GENETIC_PERT_TYPES
    ])
    if not sigs:
        return pd.Series(dtype=float), {"n_found": 0, "n_used": 0, "cell_mode": "none"}

    neural_recs = [s for s in sigs if str((s.get("meta") or {}).get("cell_line", "")).upper() in NEURAL_CELLS]
    if prefer_neural and len(neural_recs) >= 2:
        chosen = neural_recs[:max_sigs]
        cell_mode = "neural"
    else:
        chosen = sigs[:max_sigs]
        cell_mode = "all_cell_lines"

    # Primary path: download per-signature TSV via S3 persistent_id URL
    vectors = []
    for s in chosen:
        pid_url = (s.get("meta") or {}).get("persistent_id") or s.get("persistent_id")
        if pid_url and str(pid_url).startswith("http"):
            try:
                v = parse_persistent_tsv(pid_url)
                if not v.empty:
                    vectors.append(v)
            except Exception:
                continue

    if vectors:
        mat = pd.concat(vectors, axis=1)
        consensus = mat.median(axis=1)
        return consensus, {"n_found": len(sigs), "n_used": len(vectors), "cell_mode": cell_mode}

    # Fallback path: SigCom Data API /fetch
    try:
        sig_ids = [s["id"] for s in chosen if "id" in s]
        data_resp = SESSION.post(f"{SIGCOM_DATA}/fetch", json={"signatures": sig_ids}, timeout=90).json()
        if isinstance(data_resp, dict) and "data" in data_resp:
            mat = pd.DataFrame(data_resp["data"])
            mat.index = mat.index.astype(str).str.upper()
            return mat.median(axis=1), {"n_found": len(sigs), "n_used": mat.shape[1], "cell_mode": f"{cell_mode}_data_api"}
    except Exception:
        pass

    return pd.Series(dtype=float), {"n_found": len(sigs), "n_used": 0, "cell_mode": "fetch_failed"}

def zscore_series(s):
    std = s.std(ddof=0)
    if pd.isna(std) or std == 0:
        return s
    return (s - s.mean()) / std

print(f"Fetching LINCS L1000 consensus signatures (PREFER_NEURAL_CELLS={PREFER_NEURAL_CELLS}, MAX_SIGS={MAX_SIGS_PER_COMPOUND})...")
l1000_sigs = {}
sig_meta_rows = []
for label, spec in COMPOUNDS.items():
    primary_name = spec["names"][0]
    vec, info = fetch_l1000_consensus(spec)
    sig_meta_rows.append({"agent": label, "query": primary_name, **info, "n_genes": len(vec)})
    if not vec.empty:
        l1000_sigs[label] = zscore_series(vec)
        print(f"  [OK] {label:10s} ({primary_name}): {len(vec):,} genes from {info['n_used']}/{info['n_found']} sigs [{info['cell_mode']}]")
    else:
        print(f"  [WARN] {label:10s} ({primary_name}): no signature loaded (found={info['n_found']})")
    time.sleep(0.5)

sig_meta_df = pd.DataFrame(sig_meta_rows)

# Automatic single-pass recovery for any transient timeout/503 agents
timed_out = sig_meta_df[sig_meta_df["cell_mode"] == "timeout_or_error"]["agent"].tolist()
if timed_out:
    print(f"\n[RETRY] Recovering timed-out agents after brief cooldown: {timed_out}")
    time.sleep(5)
    for label in timed_out:
        spec = COMPOUNDS[label]
        vec, info = fetch_l1000_consensus(spec, limit=200)
        sig_meta_df.loc[sig_meta_df["agent"] == label, ["n_found", "n_used", "cell_mode", "n_genes"]] = [
            info["n_found"], info["n_used"], info["cell_mode"], len(vec)
        ]
        if not vec.empty:
            l1000_sigs[label] = zscore_series(vec)
            print(f"  [OK] Recovered {label:10s}: {len(vec):,} genes from {info['n_used']}/{info['n_found']} sigs [{info['cell_mode']}]")
        else:
            print(f"  [WARN] Retry failed for {label:10s} (found={info['n_found']})")

sig_meta_df.to_csv("results/results_l1000_signature_inventory.csv", index=False)
display(sig_meta_df)

if not l1000_sigs:
    raise RuntimeError("No LINCS L1000 signatures were retrieved. Check network/SigCom LINCS availability.")

sig_df = pd.DataFrame(l1000_sigs).dropna(how="all")
cgr_only_cols = [c for c in sig_df.columns if c != "Ketamine"]
if cgr_only_cols:
    sig_df["CGR_Composite"] = zscore_series(sig_df[cgr_only_cols].mean(axis=1))
cgr_no_fsk = [c for c in cgr_only_cols if c != "Forskolin"]
if "Forskolin" in cgr_only_cols and len(cgr_no_fsk) >= 2:
    sig_df["CGR_NoForskolin"] = zscore_series(sig_df[cgr_no_fsk].mean(axis=1))

sig_df.to_csv("results/l1000_consensus_matrix.csv.gz", compression="gzip")
print(f"[OK] Built z-scored consensus signature matrix: {sig_df.shape[0]:,} genes x {sig_df.shape[1]} agents.")

Fetching LINCS L1000 consensus signatures (PREFER_NEURAL_CELLS=False, MAX_SIGS=60)...
  [OK] DXM        (dextrorphan): 12,327 genes from 11/12 sigs [all_cell_lines]
  [OK] Piracetam  (piracetam): 12,327 genes from 59/89 sigs [all_cell_lines]
  [OK] CYP2D6i    (bupropion): 12,327 genes from 60/212 sigs [all_cell_lines]
  [OK] Forskolin  (forskolin): 12,327 genes from 59/250 sigs [all_cell_lines]
  [OK] Fisetin    (fisetin): 12,327 genes from 1/2 sigs [all_cell_lines]
  [WARN] NAC        (acetylcysteine): no signature loaded (found=0)
  [WARN] DHA        (docosahexaenoic acid): no signature loaded (found=0)
  [WARN] Citicoline (citicoline): no signature loaded (found=0)
  [OK] Glutamine  (glutamine): 12,327 genes from 53/54 sigs [all_cell_lines]
  [OK] Ketamine   (ketamine): 12,327 genes from 42/43 sigs [all_cell_lines]


,agent,query,n_found,n_used,cell_mode,n_genes
0,DXM,dextrorphan,12,11,all_cell_lines,12327
1,Piracetam,piracetam,89,59,all_cell_lines,12327
2,CYP2D6i,bupropion,212,60,all_cell_lines,12327
3,Forskolin,forskolin,250,59,all_cell_lines,12327
4,Fisetin,fisetin,2,1,all_cell_lines,12327
5,NAC,acetylcysteine,0,0,none,0
6,DHA,docosahexaenoic acid,0,0,none,0
7,Citicoline,citicoline,0,0,none,0
8,Glutamine,glutamine,54,53,all_cell_lines,12327
9,Ketamine,ketamine,43,42,all_cell_lines,12327


[OK] Built z-scored consensus signature matrix: 12,327 genes x 9 agents.


In [9]:
# @title 4. Score Disease Reversal, Ketamine Mimicry, & Pairwise Antagonism + Download Zip
module_genes_up = set(g.upper() for g in module_genes)

def load_deg_flexible(path):
    df = pd.read_csv(path)
    cmap = {c.lower().strip(): c for c in df.columns}
    sym_c = next((cmap[k] for k in ["symbol", "gene", "gene_symbol", "hgnc_symbol", "gene_name"] if k in cmap), None)
    fc_c  = next((cmap[k] for k in ["log2fc", "logfc", "log2foldchange", "beta", "z", "t", "stat"] if k in cmap), None)
    if sym_c is None or fc_c is None:
        raise ValueError(f"{path} needs symbol and log2FC columns; found {list(df.columns)}")
    out = df[[sym_c, fc_c]].copy()
    out.columns = ["symbol", "log2FC"]
    out["symbol"] = out["symbol"].astype(str).str.strip().str.upper()
    out["log2FC"] = pd.to_numeric(out["log2FC"], errors="coerce")
    return out.dropna().groupby("symbol", as_index=False)["log2FC"].mean()

def reversal_metrics(drug_vec, deg_df, gene_subset):
    sub = deg_df[deg_df["symbol"].isin(gene_subset)].set_index("symbol")["log2FC"]
    common = drug_vec.index.intersection(sub.index)
    if len(common) < 10:
        return np.nan, np.nan, len(common)
    d_v = drug_vec.loc[common].values
    g_v = sub.loc[common].values
    if np.all(d_v == 0) or np.all(g_v == 0):
        return np.nan, np.nan, len(common)
    cos_sim = 1.0 - cosine(d_v, g_v)
    rho, _ = spearmanr(d_v, g_v)
    return round(float(cos_sim), 4), round(float(rho), 4), int(len(common))

# 1) Module-restricted Disease Reversal (runs for all available DEG files)
DISORDER_FILES = {
    "MDD": "data/deg_MDD.csv",
    "BIP": "data/deg_BIP.csv",
    "SCZ": "data/deg_SCZ.csv",
    "OCD": "data/deg_OCD.csv",
}
rev_rows = []
for disorder, path in DISORDER_FILES.items():
    if not Path(path).exists():
        print(f"  [SKIP] {path} not found.")
        continue
    deg = load_deg_flexible(path)
    for col in sig_df.columns:
        cos_sim, rho, n_g = reversal_metrics(sig_df[col].dropna(), deg, module_genes_up)
        rev_rows.append({
            "disorder":        disorder,
            "agent":           col,
            "n_module_genes":  n_g,
            "reversal_cosine": cos_sim,
            "reversal_rho":    rho,
        })

rev_df = pd.DataFrame(rev_rows)
if not rev_df.empty:
    rev_df.to_csv("results/results_disease_reversal.csv", index=False)
    print("\n=== Module-Restricted Disease Reversal (more negative = stronger reversal) ===")
    display(rev_df.sort_values(["disorder", "reversal_cosine"]))

# 2) Ketamine-Mimicry & Pairwise Component Antagonism Matrix (inside CGR synaptic modules)
mod_sig_df = sig_df.loc[sig_df.index.intersection(module_genes_up)].dropna(how="all")
agents = list(mod_sig_df.columns)
sim_mat = pd.DataFrame(index=agents, columns=agents, dtype=float)
for a in agents:
    for b in agents:
        if a == b:
            col_vals = mod_sig_df[a].dropna()
            sim_mat.loc[a, b] = 1.0 if len(col_vals) >= 10 and not np.all(col_vals == 0) else np.nan
            continue
        pair = mod_sig_df[[a, b]].dropna()
        v1 = pair.iloc[:, 0].to_numpy()
        v2 = pair.iloc[:, 1].to_numpy()
        if len(pair) >= 10 and not np.all(v1 == 0) and not np.all(v2 == 0):
            sim_mat.loc[a, b] = round(float(1.0 - cosine(v1, v2)), 4)
        else:
            sim_mat.loc[a, b] = np.nan

sim_mat.to_csv("results/results_ketamine_and_antagonism_matrix.csv")
print(f"\n=== Ketamine-Mimicry & Pairwise Component Similarity ({len(mod_sig_df)} module genes) ===")
display(sim_mat)

# Flag strong antagonistic pairs (< -0.25)
antag_pairs = []
for i, a in enumerate(agents):
    for b in agents[i+1:]:
        val = sim_mat.loc[a, b]
        if pd.notna(val) and val < -0.25:
            antag_pairs.append({"agent_A": a, "agent_B": b, "cosine_sim": val})
if antag_pairs:
    antag_df = pd.DataFrame(antag_pairs).sort_values("cosine_sim")
    antag_df.to_csv("results/results_antagonistic_pairs.csv", index=False)
    print("\n=== Flagged Transcriptional Cancellation Pairs (cosine < -0.25) ===")
    display(antag_df)

# 3) Optional External Neuronal/iPSC Ketamine DEG Check
ext_path = Path("data/deg_ketamine_ext.csv")
if ext_path.exists():
    ket_ext = load_deg_flexible(ext_path)
    ext_rows = []
    for col in sig_df.columns:
        cos_sim, rho, n_g = reversal_metrics(sig_df[col].dropna(), ket_ext, module_genes_up)
        ext_rows.append({"agent": col, "ext_ketamine_cosine": cos_sim, "ext_ketamine_rho": rho, "n_genes": n_g})
    ext_df = pd.DataFrame(ext_rows).sort_values("ext_ketamine_cosine", ascending=False)
    ext_df.to_csv("results/results_external_ketamine_mimicry.csv", index=False)
    print("\n=== External Ketamine Mimicry ===")
    display(ext_df)
else:
    print("\n[INFO] data/deg_ketamine_ext.csv not found; skipped external ketamine benchmark.")

# Bundle & download results
zip_path = Path("cgr_transcriptomic_benchmark_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path("results").glob("*.csv*")):
        zf.write(p, arcname=p.name)

print(f"\n[OK] Bundled all outputs into {zip_path}")
from google.colab import files
files.download(str(zip_path))


=== Module-Restricted Disease Reversal (more negative = stronger reversal) ===


,disorder,agent,n_module_genes,reversal_cosine,reversal_rho
13,BIP,Fisetin,88,-0.1556,-0.0804
9,BIP,DXM,88,-0.1022,-0.0460
15,BIP,Ketamine,88,-0.1012,-0.1438
17,BIP,CGR_NoForskolin,88,-0.0141,-0.0329
11,BIP,CYP2D6i,88,-0.0135,-0.0434
16,BIP,CGR_Composite,88,0.0695,0.0373
14,BIP,Glutamine,88,0.0949,0.0176
10,BIP,Piracetam,88,0.1612,0.1594
12,BIP,Forskolin,88,0.2145,0.1020
5,MDD,Glutamine,91,-0.2313,-0.1540



=== Ketamine-Mimicry & Pairwise Component Similarity (93 module genes) ===


,DXM,Piracetam,CYP2D6i,Forskolin,Fisetin,Glutamine,Ketamine,CGR_Composite,CGR_NoForskolin
DXM,1.0000,0.2919,0.0372,-0.3186,0.0563,-0.2210,0.1601,0.3297,0.4627
Piracetam,0.2919,1.0000,0.3573,-0.1397,0.0542,0.0236,0.1844,0.5887,0.6543
CYP2D6i,0.0372,0.3573,1.0000,0.2839,-0.1677,0.2635,0.1026,0.6715,0.5687
Forskolin,-0.3186,-0.1397,0.2839,1.0000,-0.4141,0.2068,0.0462,0.2374,-0.1593
Fisetin,0.0563,0.0542,-0.1677,-0.4141,1.0000,-0.0473,-0.1043,0.2104,0.3797
Glutamine,-0.2210,0.0236,0.2635,0.2068,-0.0473,1.0000,-0.1677,0.5136,0.4391
Ketamine,0.1601,0.1844,0.1026,0.0462,-0.1043,-0.1677,1.0000,0.0718,0.0544
CGR_Composite,0.3297,0.5887,0.6715,0.2374,0.2104,0.5136,0.0718,1.0000,0.9212
CGR_NoForskolin,0.4627,0.6543,0.5687,-0.1593,0.3797,0.4391,0.0544,0.9212,1.0000



=== Flagged Transcriptional Cancellation Pairs (cosine < -0.25) ===


,agent_A,agent_B,cosine_sim
1,Forskolin,Fisetin,-0.4141
0,DXM,Forskolin,-0.3186



[INFO] data/deg_ketamine_ext.csv not found; skipped external ketamine benchmark.

[OK] Bundled all outputs into cgr_transcriptomic_benchmark_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# @title 5. Per-Agent Gene-Driver & Leave-One-Out (LOO) Attribution for Key Synaptic Genes
from pathlib import Path
import numpy as np
import pandas as pd

# Load consensus matrix from memory or disk
if "sig_df" not in globals() or sig_df is None or sig_df.empty:
    mat_path = Path("results/l1000_consensus_matrix.csv.gz")
    if not mat_path.exists():
        raise FileNotFoundError(f"Could not find {mat_path}. Run Cell 3 first.")
    sig_df = pd.read_csv(mat_path, index_col=0)

# Identify individual CGR agents (excluding Ketamine and composites)
composite_cols = {"Ketamine", "CGR_Composite", "CGR_NoForskolin"}
cgr_agents = [c for c in sig_df.columns if c not in composite_cols]
display_cols = cgr_agents + [c for c in ["CGR_Composite", "CGR_NoForskolin", "Ketamine"] if c in sig_df.columns]

KEY_GENES = [
    # Spotlight downregulated / Ketamine-opposing / BIP-SCZ-MDD penalizing genes
    "CDC42", "GRIA2", "SLC1A1", "SLC38A2", "CDKN1A", "BDNF", "EIF4EBP1", "ATF4",
    # Spotlight upregulated / glial / PI3K-Akt-mTOR & spine-consolidation genes
    "GLUL", "SLC1A3", "SLC1A2", "PIK3CA", "AKT3", "GLS2", "ACTB", "NLGN1", "WASL", "ADCY1", "NTRK2",
]
present_genes = [g for g in KEY_GENES if g in sig_df.index]

# 1) Gene x Agent Z-score matrix + primary driver attribution
sub = sig_df.loc[present_genes, display_cols].copy()

driver_rows = []
for g in present_genes:
    agent_vals = sig_df.loc[g, cgr_agents].dropna()
    comp_val = sig_df.loc[g, "CGR_Composite"] if "CGR_Composite" in sig_df.columns else agent_vals.mean()
    direction = np.sign(comp_val) if pd.notna(comp_val) and comp_val != 0 else 1.0

    # Same-sign agents pushing the composite in its observed direction
    same_sign = agent_vals[agent_vals * direction > 0]
    if not same_sign.empty:
        top_agent = same_sign.abs().idxmax()
        top_val = float(agent_vals[top_agent])
        share_pct = 100.0 * abs(top_val) / same_sign.abs().sum()
    else:
        top_agent, top_val, share_pct = "None", np.nan, np.nan

    # Strongest opposing agent (buffering the shift)
    opp_sign = agent_vals[agent_vals * direction < 0]
    if not opp_sign.empty:
        opp_agent = opp_sign.abs().idxmax()
        opp_val = float(agent_vals[opp_agent])
    else:
        opp_agent, opp_val = "None", np.nan

    driver_rows.append({
        "gene": g,
        "CGR_dir": "UP" if direction > 0 else "DOWN",
        "Primary_Driver": f"{top_agent} ({top_val:+.2f})",
        "Driver_Share_%": round(share_pct, 1) if pd.notna(share_pct) else np.nan,
        "Top_Counterweight": f"{opp_agent} ({opp_val:+.2f})" if opp_agent != "None" else "None",
    })

driver_summary = pd.DataFrame(driver_rows).set_index("gene")
full_table = pd.concat([sub.round(3), driver_summary], axis=1)
full_table.to_csv("results/results_per_agent_gene_drivers.csv")

print("=== Per-Agent Z-Scores & Primary Drivers for Key Synaptic Module Genes ===")
display(full_table)

# 2) Leave-One-Out (LOO) shift for CDC42, GRIA2, and SLC1A1
spotlight = [g for g in ["CDC42", "GRIA2", "SLC1A1", "GLUL", "PIK3CA"] if g in sig_df.index]
loo_rows = []
for g in spotlight:
    vals = sig_df.loc[g, cgr_agents].dropna()
    base_mean = float(vals.mean())
    row = {"gene": g, "Raw_Mean_All": round(base_mean, 3)}
    for ag in cgr_agents:
        if ag in vals.index and len(vals) > 1:
            loo_mean = float(vals.drop(ag).mean())
            delta = loo_mean - base_mean  # positive delta = dropping this agent makes gene LESS negative / MORE positive
            row[f"Drop_{ag}"] = f"{loo_mean:+.3f} (Δ{delta:+.3f})"
        else:
            row[f"Drop_{ag}"] = "N/A"
    loo_rows.append(row)

loo_df = pd.DataFrame(loo_rows).set_index("gene")
loo_df.to_csv("results/results_loo_spotlight_genes.csv")

print("\n=== Leave-One-Out (LOO) Mean Shift for Spotlight Genes (Δ > 0 means dropping agent lifts expression) ===")
display(loo_df)

print("\n[OK] Saved results/results_per_agent_gene_drivers.csv and results/results_loo_spotlight_genes.csv")

=== Per-Agent Z-Scores & Primary Drivers for Key Synaptic Module Genes ===


,DXM,Piracetam,CYP2D6i,Forskolin,Fisetin,Glutamine,CGR_Composite,CGR_NoForskolin,Ketamine,CGR_dir,Primary_Driver,Driver_Share_%,Top_Counterweight
CDC42,-1.364,0.184,0.929,0.228,-0.702,-0.864,-0.654,-0.777,3.035,DOWN,DXM (-1.36),46.6,CYP2D6i (+0.93)
GRIA2,-3.389,-0.884,-0.543,4.096,-1.172,0.552,-0.552,-2.325,-0.820,DOWN,DXM (-3.39),56.6,Forskolin (+4.10)
SLC1A1,-2.008,-1.704,0.396,2.579,-1.727,0.289,-0.895,-2.032,1.500,DOWN,DXM (-2.01),36.9,Forskolin (+2.58)
SLC38A2,0.294,-2.442,-1.874,-0.963,-2.212,-0.732,-3.264,-2.979,0.771,DOWN,Piracetam (-2.44),29.7,DXM (+0.29)
CDKN1A,-0.370,-1.086,-2.077,-0.994,-0.492,-0.765,-2.381,-2.048,-2.404,DOWN,CYP2D6i (-2.08),35.9,None
BDNF,-0.447,-0.138,-0.033,-2.332,0.692,-0.281,-1.045,-0.089,-0.879,DOWN,Forskolin (-2.33),72.2,Fisetin (+0.69)
EIF4EBP1,0.659,0.456,-0.745,-0.316,-1.824,-0.403,-0.895,-0.794,0.805,DOWN,Fisetin (-1.82),55.5,DXM (+0.66)
ATF4,1.459,-1.928,-1.402,0.307,-0.493,-1.432,-1.436,-1.623,-0.808,DOWN,Piracetam (-1.93),36.7,DXM (+1.46)
GLUL,-2.503,1.169,3.328,3.386,-2.023,4.601,3.276,1.955,-0.696,UP,Glutamine (+4.60),36.9,DXM (-2.50)
SLC1A3,-0.541,0.321,-0.298,1.049,-0.462,0.623,0.285,-0.153,-0.168,UP,Forskolin (+1.05),52.6,DXM (-0.54)



=== Leave-One-Out (LOO) Mean Shift for Spotlight Genes (Δ > 0 means dropping agent lifts expression) ===


,Raw_Mean_All,Drop_DXM,Drop_Piracetam,Drop_CYP2D6i,Drop_Forskolin,Drop_Fisetin,Drop_Glutamine
gene,,,,,,,
CDC42,-0.265,-0.045 (Δ+0.220),-0.355 (Δ-0.090),-0.504 (Δ-0.239),-0.363 (Δ-0.099),-0.177 (Δ+0.087),-0.145 (Δ+0.120)
GRIA2,-0.224,+0.410 (Δ+0.633),-0.091 (Δ+0.132),-0.160 (Δ+0.064),-1.087 (Δ-0.864),-0.034 (Δ+0.190),-0.379 (Δ-0.155)
SLC1A1,-0.362,-0.033 (Δ+0.329),-0.094 (Δ+0.268),-0.514 (Δ-0.152),-0.951 (Δ-0.588),-0.090 (Δ+0.273),-0.493 (Δ-0.130)
GLUL,1.326,+2.092 (Δ+0.766),+1.358 (Δ+0.032),+0.926 (Δ-0.400),+0.914 (Δ-0.412),+1.996 (Δ+0.670),+0.672 (Δ-0.655)
PIK3CA,0.929,+1.089 (Δ+0.160),+1.201 (Δ+0.272),+0.999 (Δ+0.070),+1.220 (Δ+0.291),+1.086 (Δ+0.157),-0.020 (Δ-0.950)



[OK] Saved results/results_per_agent_gene_drivers.csv and results/results_loo_spotlight_genes.csv


In [11]:
# @title 6. Head-to-Head CYP2D6 Inhibitor Benchmark (Bupropion, Paroxetine, Duloxetine, Quinidine, Fluoxetine)
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine

# Ensure Cell 3 & Cell 4 helpers are loaded
for fn in ["fetch_l1000_consensus", "zscore_series", "load_deg_flexible", "reversal_metrics"]:
    if fn not in globals():
        raise RuntimeError(f"Missing `{fn}` in workspace. Run Cells 1, 3, and 4 first.")

# Load consensus matrix if needed
if "sig_df" not in globals() or sig_df is None or sig_df.empty:
    sig_df = pd.read_csv("results/l1000_consensus_matrix.csv.gz", index_col=0)

CYP2D6_CANDIDATES = {
    "bupropion": {
        "names": ["bupropion", "bupropion hydrochloride", "amfebutamone"],
        "pert_ids": [],
    },
    "paroxetine": {
        "names": ["paroxetine", "paroxetine hydrochloride", "paroxetine hemihydrate"],
        "pert_ids": [],
    },
    "duloxetine": {
        "names": ["duloxetine", "duloxetine hydrochloride"],
        "pert_ids": [],
    },
    "quinidine": {
        "names": ["quinidine", "quinidine sulfate", "quinidine gluconate", "quinidine hydrochloride"],
        "pert_ids": [],
    },
    "fluoxetine": {
        "names": ["fluoxetine", "fluoxetine hydrochloride"],
        "pert_ids": ["BRD-A31159102"],
    },
}

SPOTLIGHT_GENES = ["CDC42", "SLC1A3", "GLUL", "AKT3", "WASL", "GRIA2", "SLC1A1", "NLGN1", "BDNF"]
DISORDERS_TO_SCORE = ["MDD", "BIP", "SCZ"]  # OCD has only 5 overlapping module genes

# Load available postmortem DEG tables
loaded_degs = {}
for dis in DISORDERS_TO_SCORE:
    p = Path(DISORDER_FILES[dis])
    if p.exists():
        loaded_degs[dis] = load_deg_flexible(p)

def mod_cosine(v1, v2, gene_set):
    s1 = v1.reindex(list(gene_set)).dropna()
    s2 = v2.reindex(list(gene_set)).dropna()
    common = s1.index.intersection(s2.index)
    if len(common) < 10:
        return np.nan
    a, b = s1.loc[common].to_numpy(dtype=float), s2.loc[common].to_numpy(dtype=float)
    if np.all(a == 0) or np.all(b == 0):
        return np.nan
    return round(float(1.0 - cosine(a, b)), 4)

# 1) Fetch & z-score all 5 CYP2D6 inhibitors
cyp_vecs = {}
cyp_meta = {}
print("Fetching LINCS L1000 consensus signatures for CYP2D6 inhibitors...")
for name, spec in CYP2D6_CANDIDATES.items():
    vec, info = fetch_l1000_consensus(spec, prefer_neural=PREFER_NEURAL_CELLS, max_sigs=MAX_SIGS_PER_COMPOUND)
    cyp_meta[name] = info
    if not vec.empty:
        cyp_vecs[name] = zscore_series(vec)
        print(f"  [OK] {name:11s}: {len(vec):,} genes | {info['n_used']}/{info['n_found']} sigs [{info['cell_mode']}]")
    else:
        print(f"  [WARN] {name:11s}: no signatures retrieved (found={info['n_found']})")

ket_vec = sig_df["Ketamine"].dropna() if "Ketamine" in sig_df.columns else pd.Series(dtype=float)

# 2) Standalone Head-to-Head Table
agent_rows = []
for name, vec in cyp_vecs.items():
    info = cyp_meta[name]
    row = {
        "CYP2D6i": name,
        "sigs_used": f"{info['n_used']}/{info['n_found']}",
        "cell_mode": info["cell_mode"],
        "Ketamine_cos": mod_cosine(vec, ket_vec, module_genes_up) if not ket_vec.empty else np.nan,
    }
    rev_vals = []
    for dis, deg in loaded_degs.items():
        cos_v, _, _ = reversal_metrics(vec, deg, module_genes_up)
        row[f"{dis}_cos"] = cos_v
        if pd.notna(cos_v):
            rev_vals.append(cos_v)
    row["Mean_Rev_cos"] = round(float(np.mean(rev_vals)), 4) if rev_vals else np.nan
    for g in SPOTLIGHT_GENES:
        row[g] = round(float(vec[g]), 3) if g in vec.index else np.nan
    agent_rows.append(row)

agent_df = pd.DataFrame(agent_rows).sort_values("Mean_Rev_cos", ascending=True)
agent_df.to_csv("results/results_cyp2d6i_head2head_agents.csv", index=False)

print("\n=== A. Standalone CYP2D6 Inhibitor Head-to-Head (sorted by Mean_Rev_cos; more negative = stronger reversal) ===")
display(agent_df)

# 3) Swap-In Composite Impact (both CGR_Composite & CGR_NoForskolin)
base_cgr_cols = [c for c in sig_df.columns if c not in {"Ketamine", "CGR_Composite", "CGR_NoForskolin", "CYP2D6i"}]
base_nofsk_cols = [c for c in base_cgr_cols if c != "Forskolin"]

swap_rows = []
for stack_name, cols in [("CGR_Composite", base_cgr_cols), ("CGR_NoForskolin", base_nofsk_cols)]:
    for name, vec in cyp_vecs.items():
        tmp = sig_df[cols].copy()
        tmp["CYP2D6i"] = vec.reindex(tmp.index)
        comp = zscore_series(tmp.mean(axis=1).dropna())

        row = {
            "Stack": stack_name,
            "CYP2D6i": name,
            "Ketamine_cos": mod_cosine(comp, ket_vec, module_genes_up) if not ket_vec.empty else np.nan,
        }
        rev_vals = []
        for dis, deg in loaded_degs.items():
            cos_v, _, _ = reversal_metrics(comp, deg, module_genes_up)
            row[f"{dis}_cos"] = cos_v
            if pd.notna(cos_v):
                rev_vals.append(cos_v)
        row["Mean_Rev_cos"] = round(float(np.mean(rev_vals)), 4) if rev_vals else np.nan
        for g in ["CDC42", "SLC1A3", "GLUL", "AKT3", "GRIA2", "SLC1A1", "NLGN1", "BDNF"]:
            row[g] = round(float(comp[g]), 3) if g in comp.index else np.nan
        swap_rows.append(row)

swap_df = pd.DataFrame(swap_rows).sort_values(["Stack", "Mean_Rev_cos"], ascending=[True, True])
swap_df.to_csv("results/results_cyp2d6i_swap_composites.csv", index=False)

print("\n=== B. Swap-In Stack Impact (Replacing CYP2D6i inside CGR_Composite & CGR_NoForskolin) ===")
display(swap_df)

print("\n[OK] Saved results/results_cyp2d6i_head2head_agents.csv and results/results_cyp2d6i_swap_composites.csv")

Fetching LINCS L1000 consensus signatures for CYP2D6 inhibitors...
  [OK] bupropion  : 12,327 genes | 60/212 sigs [all_cell_lines]
  [OK] paroxetine : 12,327 genes | 60/250 sigs [all_cell_lines]
  [OK] duloxetine : 12,327 genes | 60/250 sigs [all_cell_lines]
  [OK] quinidine  : 12,327 genes | 60/200 sigs [all_cell_lines]
  [OK] fluoxetine : 12,327 genes | 60/250 sigs [all_cell_lines]

=== A. Standalone CYP2D6 Inhibitor Head-to-Head (sorted by Mean_Rev_cos; more negative = stronger reversal) ===


,CYP2D6i,sigs_used,cell_mode,Ketamine_cos,MDD_cos,BIP_cos,SCZ_cos,Mean_Rev_cos,CDC42,SLC1A3,GLUL,AKT3,WASL,GRIA2,SLC1A1,NLGN1,BDNF
3,quinidine,60/200,all_cell_lines,-0.0758,-0.0790,-0.0693,-0.0320,-0.0601,0.324,-1.746,3.919,-1.135,-1.275,-1.947,2.772,1.586,-0.264
0,bupropion,60/212,all_cell_lines,0.1026,0.0640,-0.0135,-0.0150,0.0118,0.929,-0.298,3.328,-0.123,0.796,-0.543,0.396,0.875,-0.033
4,fluoxetine,60/250,all_cell_lines,-0.0270,0.0991,0.0813,0.0492,0.0765,0.413,0.101,-0.904,-0.397,1.239,-3.311,-0.894,-1.320,-0.380
1,paroxetine,60/250,all_cell_lines,0.0349,-0.0632,0.1868,0.2304,0.1180,-0.006,1.358,1.817,1.774,-0.797,-4.744,3.366,-1.178,-0.239
2,duloxetine,60/250,all_cell_lines,0.0643,0.1326,0.2909,0.2972,0.2402,-0.690,2.803,4.202,1.841,0.197,-1.124,-2.078,-0.661,0.367



=== B. Swap-In Stack Impact (Replacing CYP2D6i inside CGR_Composite & CGR_NoForskolin) ===


,Stack,CYP2D6i,Ketamine_cos,MDD_cos,BIP_cos,SCZ_cos,Mean_Rev_cos,CDC42,SLC1A3,GLUL,AKT3,GRIA2,SLC1A1,NLGN1,BDNF
3,CGR_Composite,quinidine,0.0063,0.0147,0.0532,0.0763,0.0481,-0.901,-0.310,3.510,1.231,-1.127,0.083,1.431,-1.137
0,CGR_Composite,bupropion,0.0718,0.0664,0.0695,0.0763,0.0707,-0.654,0.285,3.276,1.651,-0.552,-0.895,1.142,-1.045
4,CGR_Composite,fluoxetine,0.0266,0.1001,0.1295,0.1223,0.1173,-0.906,0.469,1.604,1.607,-1.768,-1.490,0.250,-1.241
1,CGR_Composite,paroxetine,0.0532,0.0158,0.1693,0.1981,0.1277,-1.110,1.032,2.835,2.597,-2.437,0.350,0.317,-1.207
2,CGR_Composite,duloxetine,0.0629,0.1028,0.2051,0.2154,0.1744,-1.350,1.596,3.718,2.515,-0.809,-1.957,0.521,-0.900
8,CGR_NoForskolin,quinidine,-0.0135,-0.0492,-0.0375,-0.0327,-0.0398,-1.043,-0.777,2.224,1.000,-2.946,-1.024,1.848,-0.188
5,CGR_NoForskolin,bupropion,0.0544,0.0077,-0.0141,-0.0242,-0.0102,-0.777,-0.153,1.955,1.426,-2.325,-2.032,1.531,-0.089
9,CGR_NoForskolin,fluoxetine,0.0045,0.0270,0.0266,0.0024,0.0187,-1.033,0.018,0.151,1.355,-3.635,-2.677,0.614,-0.245
6,CGR_NoForskolin,paroxetine,0.0324,-0.0461,0.0771,0.0875,0.0395,-1.270,0.599,1.413,2.414,-4.448,-0.823,0.705,-0.190
7,CGR_NoForskolin,duloxetine,0.0433,0.0422,0.1185,0.1119,0.0909,-1.510,1.206,2.394,2.329,-2.645,-3.176,0.899,0.085



[OK] Saved results/results_cyp2d6i_head2head_agents.csv and results/results_cyp2d6i_swap_composites.csv


In [12]:
# @title 7. Generate Master Detailed Summary Report (.txt) & Update Download Bundle
from datetime import datetime, timezone
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr

summary_path = Path("results/cgr_transcriptomic_benchmark_master_summary.txt")

def _hr(title="", char="=", width=96):
    if not title:
        return char * width
    return f"\n{char * width}\n{title}\n{char * width}"

def _safe_cos_rho(s1, s2, min_n=5):
    pair = pd.concat([s1.rename("a"), s2.rename("b")], axis=1).dropna()
    if len(pair) < min_n:
        return np.nan, np.nan, len(pair)
    v1 = pair["a"].to_numpy(dtype=float)
    v2 = pair["b"].to_numpy(dtype=float)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return np.nan, np.nan, len(pair)
    return round(float(1.0 - cosine(v1, v2)), 4), round(float(spearmanr(v1, v2)[0]), 4), int(len(pair))

def _load_or_var(var_name, csv_path, index_col=None):
    obj = globals().get(var_name)
    if isinstance(obj, pd.DataFrame) and not obj.empty:
        return obj.copy()
    p = Path(csv_path)
    if p.exists():
        return pd.read_csv(p, index_col=index_col)
    return pd.DataFrame()

# Recover core tables from memory or disk
sig_df_r      = _load_or_var("sig_df", "results/l1000_consensus_matrix.csv.gz", index_col=0)
sig_meta_r    = _load_or_var("sig_meta_df", "results/results_l1000_signature_inventory.csv")
rev_df_r      = _load_or_var("rev_df", "results/results_disease_reversal.csv")
sim_mat_r     = _load_or_var("sim_mat", "results/results_ketamine_and_antagonism_matrix.csv", index_col=0)
ext_df_r      = _load_or_var("ext_df", "results/results_external_ketamine_mimicry.csv")
drivers_df_r  = _load_or_var("full_table", "results/results_per_agent_gene_drivers.csv", index_col=0)
loo_df_r      = _load_or_var("loo_df", "results/results_loo_spotlight_genes.csv", index_col=0)
cyp_agents_r  = _load_or_var("agent_df", "results/results_cyp2d6i_head2head_agents.csv")
cyp_swaps_r   = _load_or_var("swap_df", "results/results_cyp2d6i_swap_composites.csv")

mod_sig_r = sig_df_r.loc[sig_df_r.index.intersection(module_genes_up)].dropna(how="all") if not sig_df_r.empty else pd.DataFrame()

lines = []
lines.append("=" * 96)
lines.append("CGR TRANSCRIPTOMIC BENCHMARK — MASTER DETAILED ANALYSIS SUMMARY")
lines.append("=" * 96)
lines.append(f"Generated (UTC)         : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
lines.append(f"Primary CYP2D6 Inhibitor: {CYP2D6_INHIBITOR}")
lines.append(f"Prefer Neural Cells     : {PREFER_NEURAL_CELLS}")
lines.append(f"Include Adhesion/PSD    : {INCLUDE_ADHESION_PSD}")
lines.append(f"Max Sigs per Compound   : {MAX_SIGS_PER_COMPOUND}")
lines.append(f"Total Anchor Genes      : {len(all_anchor_genes)}")
lines.append(f"Target Module Genes     : {len(module_genes)}")

# -------------------------------------------------------------------------
# 1. SYNAPTIC MODULE & LINCS L1000 GENE COVERAGE
# -------------------------------------------------------------------------
lines.append(_hr("1. SYNAPTIC MODULE & LINCS L1000 GENE COVERAGE"))
measured_genes = set(sig_df_r.index)
covered_mod = sorted(module_genes_up & measured_genes)
missing_mod = sorted(module_genes_up - measured_genes)
lines.append(f"Consensus L1000 Matrix  : {sig_df_r.shape[0]:,} total genes x {sig_df_r.shape[1]} columns")
lines.append(f"Target Module Coverage  : {len(covered_mod)} / {len(module_genes_up)} genes ({100*len(covered_mod)/max(1,len(module_genes_up)):.1f}%)")
if missing_mod:
    lines.append(f"Missing Module Genes ({len(missing_mod)}): {', '.join(missing_mod)}")

sub_modules = ["LTP_init", "TrkB_mTOR", "Actin_spine"] + (["Adhesion_PSD"] if INCLUDE_ADHESION_PSD else [])
mod_map = {}
for m in sub_modules:
    genes_m = set(
        anchors[anchors["module"].fillna("").str.contains(m, case=False, regex=False)]["symbol"]
        .dropna().str.upper()
    )
    mod_map[m] = genes_m
    cov_m = len(genes_m & measured_genes)
    lines.append(f"  - {m:14s}: {len(genes_m):3d} defined | {cov_m:3d} covered in L1000 ({100*cov_m/max(1,len(genes_m)):.1f}%)")

# -------------------------------------------------------------------------
# 2. LINCS L1000 SIGNATURE INVENTORY (CORE STACK)
# -------------------------------------------------------------------------
lines.append(_hr("2. LINCS L1000 SIGNATURE INVENTORY (CELL 3 CORE STACK)"))
if not sig_meta_r.empty:
    lines.append(sig_meta_r.to_string(index=False))
    failed_agents = sig_meta_r[sig_meta_r["n_used"] == 0]["agent"].tolist()
    low_n_agents = sig_meta_r[(sig_meta_r["n_used"] > 0) & (sig_meta_r["n_used"] < 5)]["agent"].tolist()
    if failed_agents:
        lines.append(f"\n[MISSING] Agents with 0 retrieved signatures : {', '.join(failed_agents)}")
    if low_n_agents:
        lines.append(f"[LOW-N]   Agents with < 5 signatures used    : {', '.join(low_n_agents)}")

# -------------------------------------------------------------------------
# 3. POSTMORTEM BRAIN DEG INVENTORY & MODULE-RESTRICTED DISEASE REVERSAL
# -------------------------------------------------------------------------
lines.append(_hr("3. POSTMORTEM BRAIN DISEASE REVERSAL (MORE NEGATIVE = STRONGER REVERSAL)"))
all_loaded_degs = {}
for disorder, path in DISORDER_FILES.items():
    if Path(path).exists():
        d_df = load_deg_flexible(path)
        all_loaded_degs[disorder] = d_df
        overlap_m = len(set(d_df["symbol"]) & module_genes_up)
        overlap_l1000 = len(set(d_df["symbol"]) & set(covered_mod))
        flag = "  [UNDERPOWERED <10]" if overlap_l1000 < 10 else ""
        lines.append(f"  - {disorder:4s} ({path}): {len(d_df):,} total genes | {overlap_m} module genes | {overlap_l1000} shared w/ L1000{flag}")
    else:
        lines.append(f"  - {disorder:4s} ({path}): NOT FOUND (skipped)")

if not rev_df_r.empty:
    for disorder in sorted(rev_df_r["disorder"].unique()):
        lines.append(f"\n--- {disorder} Reversal Ranking (sorted by reversal_cosine ascending) ---")
        sub = rev_df_r[rev_df_r["disorder"] == disorder].sort_values("reversal_cosine", ascending=True)
        lines.append(sub.to_string(index=False))

    lines.append("\n--- Cross-Disorder Mean Reversal Summary ---")
    cross = (
        rev_df_r.groupby("agent", as_index=False)
        .agg(
            disorders_scored=("reversal_cosine", lambda s: int(s.notna().sum())),
            mean_cosine=("reversal_cosine", "mean"),
            median_cosine=("reversal_cosine", "median"),
            mean_rho=("reversal_rho", "mean"),
            best_disorder=("reversal_cosine", lambda s: rev_df_r.loc[s.idxmin(), "disorder"] if s.notna().any() else "N/A"),
            best_cosine=("reversal_cosine", "min"),
        )
        .sort_values("mean_cosine", ascending=True)
    )
    for col in ["mean_cosine", "median_cosine", "mean_rho", "best_cosine"]:
        cross[col] = cross[col].round(4)
    lines.append(cross.to_string(index=False))

    # Head-to-head: CGR_Composite vs CGR_NoForskolin vs Ketamine
    h2h_agents = [a for a in ["CGR_Composite", "CGR_NoForskolin", "Ketamine"] if a in set(rev_df_r["agent"])]
    if len(h2h_agents) >= 2:
        lines.append(f"\n--- Head-to-Head Comparison ({' vs. '.join(h2h_agents)}) ---")
        piv_cos = rev_df_r.pivot(index="disorder", columns="agent", values="reversal_cosine")[h2h_agents]
        piv_rho = rev_df_r.pivot(index="disorder", columns="agent", values="reversal_rho")[h2h_agents]
        for dis in piv_cos.index:
            parts = []
            valid_scores = {}
            for ag in h2h_agents:
                c_v, r_v = piv_cos.loc[dis, ag], piv_rho.loc[dis, ag]
                parts.append(f"{ag}: cos={c_v:+.4f}, rho={r_v:+.4f}" if pd.notna(c_v) else f"{ag}: NaN")
                if pd.notna(c_v):
                    valid_scores[ag] = c_v
            winner = min(valid_scores, key=valid_scores.get) if valid_scores else "N/A (underpowered)"
            lines.append(f"  {dis:4s} | " + " | ".join(parts) + f" | Best: {winner}")

    # Sub-module breakdown
    lines.append("\n--- Sub-Module Reversal Breakdown (CGR_Composite, CGR_NoForskolin & Ketamine) ---")
    submod_rows = []
    for dis, d_df in all_loaded_degs.items():
        deg_s = d_df.set_index("symbol")["log2FC"]
        for m_name, m_genes in mod_map.items():
            for ag in h2h_agents:
                if ag in sig_df_r.columns:
                    c_val, r_val, n_ov = _safe_cos_rho(sig_df_r[ag].reindex(list(m_genes)), deg_s, min_n=5)
                    submod_rows.append({
                        "disorder": dis, "sub_module": m_name, "agent": ag,
                        "n_genes": n_ov, "cosine": c_val, "spearman_rho": r_val
                    })
    if submod_rows:
        lines.append(pd.DataFrame(submod_rows).to_string(index=False))

# -------------------------------------------------------------------------
# 4. KETAMINE-MIMICRY ANALYSIS
# -------------------------------------------------------------------------
lines.append(_hr("4. KETAMINE-MIMICRY ANALYSIS (WITHIN CGR SYNAPTIC MODULES)"))
if not mod_sig_r.empty and "Ketamine" in mod_sig_r.columns:
    ket_rows = []
    ket_vec = mod_sig_r["Ketamine"]
    for ag in mod_sig_r.columns:
        if ag == "Ketamine":
            continue
        c_val, r_val, n_ov = _safe_cos_rho(mod_sig_r[ag], ket_vec, min_n=10)
        ket_rows.append({"agent": ag, "n_shared_mod_genes": n_ov, "ketamine_cosine": c_val, "ketamine_rho": r_val})
    lines.append(pd.DataFrame(ket_rows).sort_values("ketamine_cosine", ascending=False).to_string(index=False))

if not ext_df_r.empty:
    lines.append("\n--- External Neuronal/iPSC Ketamine DEG Mimicry ---")
    lines.append(ext_df_r.to_string(index=False))

# -------------------------------------------------------------------------
# 5. PAIRWISE SYNERGY & ANTAGONISM (INDIVIDUAL COMPONENTS ONLY)
# -------------------------------------------------------------------------
lines.append(_hr("5. PAIRWISE COMPONENT SIMILARITY, SYNERGY & ANTAGONISM"))
if not sim_mat_r.empty:
    lines.append("Full Pairwise Cosine Similarity Matrix (inside CGR synaptic modules):")
    lines.append(sim_mat_r.to_string())

    indiv_agents = [a for a in sim_mat_r.columns if a not in {"CGR_Composite", "CGR_NoForskolin", "Ketamine"}]
    pair_rows = []
    for i, a in enumerate(indiv_agents):
        for b in indiv_agents[i+1:]:
            val = sim_mat_r.loc[a, b]
            if pd.notna(val):
                pair_rows.append({"agent_A": a, "agent_B": b, "cosine_sim": float(val)})
    pair_df = pd.DataFrame(pair_rows).sort_values("cosine_sim", ascending=False)

    if not pair_df.empty:
        lines.append("\n--- Ranked Individual CGR Component Pairs (Most Concordant to Most Divergent) ---")
        lines.append(pair_df.to_string(index=False))

        flagged = pair_df[pair_df["cosine_sim"] < -0.25].sort_values("cosine_sim")
        if not flagged.empty:
            lines.append("\n--- Flagged Cancellation Pairs (cosine < -0.25) & Top Opposing Driver Genes ---")
            for _, row in flagged.iterrows():
                a, b, val = row["agent_A"], row["agent_B"], row["cosine_sim"]
                sub_p = mod_sig_r[[a, b]].dropna().copy()
                sub_p["prod"] = sub_p[a] * sub_p[b]
                top_opp = sub_p.sort_values("prod").head(8)
                drivers = ", ".join(f"{g} ({a}:{r[a]:+.2f}, {b}:{r[b]:+.2f})" for g, r in top_opp.iterrows())
                lines.append(f"  * {a} vs {b} (cos={val:+.4f}) -> {drivers}")
        else:
            lines.append("\n[OK] No individual CGR component pair crossed the strong cancellation threshold (cosine < -0.25).")
            closest = pair_df.iloc[-1]
            lines.append(f"[INFO] Closest pair to threshold: {closest['agent_A']} vs {closest['agent_B']} (cos={closest['cosine_sim']:+.4f})")

# -------------------------------------------------------------------------
# 6. GENE-LEVEL MECHANISTIC DRIVERS INSIDE CGR SYNAPTIC MODULES
# -------------------------------------------------------------------------
lines.append(_hr("6. TOP GENE-LEVEL DRIVERS IN CGR SYNAPTIC MODULES"))
anchor_lookup = anchors.drop_duplicates("symbol").set_index(anchors.drop_duplicates("symbol")["symbol"].str.upper())

for ag in [a for a in ["CGR_Composite", "CGR_NoForskolin", "Ketamine"] if a in mod_sig_r.columns]:
    s = mod_sig_r[ag].dropna().sort_values(ascending=False)
    lines.append(f"\n--- {ag}: Top 10 Upregulated & Top 10 Downregulated Module Genes ---")
    lines.append("  UPREGULATED:")
    for g, val in s.head(10).items():
        mod_lbl = anchor_lookup.loc[g, "module"] if g in anchor_lookup.index else ""
        fam_lbl = anchor_lookup.loc[g, "functional_family"] if g in anchor_lookup.index else ""
        lines.append(f"    {g:10s} : {val:+7.3f}  [{mod_lbl}] ({fam_lbl})")
    lines.append("  DOWNREGULATED:")
    for g, val in s.tail(10).iloc[::-1].items():
        mod_lbl = anchor_lookup.loc[g, "module"] if g in anchor_lookup.index else ""
        fam_lbl = anchor_lookup.loc[g, "functional_family"] if g in anchor_lookup.index else ""
        lines.append(f"    {g:10s} : {val:+7.3f}  [{mod_lbl}] ({fam_lbl})")

for target_stack in [a for a in ["CGR_Composite", "CGR_NoForskolin"] if a in mod_sig_r.columns]:
    if all_loaded_degs:
        lines.append(f"\n--- {target_stack} Gene-Level Reversal vs. Exacerbation Drivers per Disorder ---")
        stack_s = mod_sig_r[target_stack].dropna()
        for dis, d_df in all_loaded_degs.items():
            deg_s = d_df.set_index("symbol")["log2FC"]
            comb = pd.concat([deg_s.rename("disease_log2FC"), stack_s.rename("stack_z")], axis=1).dropna()
            if comb.empty:
                continue
            comb["product"] = comb["disease_log2FC"] * comb["stack_z"]
            lines.append(f"\n  [{dis}] Top Reversed Genes (opposite sign; most negative product):")
            for g, r in comb.sort_values("product").head(6).iterrows():
                lines.append(f"    {g:10s} | disease={r['disease_log2FC']:+.3f} | {target_stack}={r['stack_z']:+.3f} | prod={r['product']:+.3f}")
            lines.append(f"  [{dis}] Top Concordant/Exacerbated Genes (same sign; most positive product):")
            for g, r in comb.sort_values("product", ascending=False).head(5).iterrows():
                lines.append(f"    {g:10s} | disease={r['disease_log2FC']:+.3f} | {target_stack}={r['stack_z']:+.3f} | prod={r['product']:+.3f}")

# -------------------------------------------------------------------------
# 7. PER-AGENT GENE-DRIVER ATTRIBUTION & LEAVE-ONE-OUT (LOO) SHIFTS (CELL 5)
# -------------------------------------------------------------------------
lines.append(_hr("7. PER-AGENT GENE-DRIVER ATTRIBUTION & LEAVE-ONE-OUT (LOO) ANALYSIS (CELL 5)"))
if not drivers_df_r.empty:
    lines.append("--- Per-Agent Z-Scores, Primary Driver & Counterweight (19 Key Synaptic Genes) ---")
    lines.append(drivers_df_r.to_string())
else:
    lines.append("[INFO] Cell 5 gene-driver table not found (run Cell 5 to include).")

if not loo_df_r.empty:
    lines.append("\n--- Leave-One-Out (LOO) Spotlight Gene Shifts (Δ > 0 means dropping agent lifts gene) ---")
    lines.append(loo_df_r.to_string())
else:
    lines.append("[INFO] Cell 5 LOO spotlight table not found.")

# -------------------------------------------------------------------------
# 8. HEAD-TO-HEAD CYP2D6 INHIBITOR BENCHMARK & SWAP-IN IMPACT (CELL 6)
# -------------------------------------------------------------------------
lines.append(_hr("8. HEAD-TO-HEAD CYP2D6 INHIBITOR BENCHMARK & SWAP-IN COMPOSITES (CELL 6)"))
if not cyp_agents_r.empty:
    lines.append("--- A. Standalone CYP2D6 Inhibitor Head-to-Head (sorted by Mean_Rev_cos) ---")
    lines.append(cyp_agents_r.to_string(index=False))
else:
    lines.append("[INFO] Cell 6 standalone CYP2D6i table not found (run Cell 6 to include).")

if not cyp_swaps_r.empty:
    lines.append("\n--- B. Swap-In Composite Impact (CGR_Composite & CGR_NoForskolin) ---")
    lines.append(cyp_swaps_r.to_string(index=False))

# -------------------------------------------------------------------------
# 9. AUTOMATED QC & METHODOLOGICAL AUDIT FLAGS
# -------------------------------------------------------------------------
lines.append(_hr("9. AUTOMATED METHODOLOGICAL & QC AUDIT FLAGS"))
qc_flags = []

# Cell-line mode mismatch check
if not sig_meta_r.empty:
    modes = sig_meta_r[sig_meta_r["n_used"] > 0].groupby("cell_mode")["agent"].apply(list).to_dict()
    if len(modes) > 1:
        mode_str = "; ".join(f"{k}: {', '.join(v)}" for k, v in modes.items())
        qc_flags.append(f"CELL-LINE MISMATCH (Core Stack): Mixed cell backgrounds detected -> {mode_str}.")
if not cyp_agents_r.empty and "cell_mode" in cyp_agents_r.columns:
    cyp_modes = cyp_agents_r.groupby("cell_mode")["CYP2D6i"].apply(list).to_dict()
    if len(cyp_modes) > 1:
        cyp_mode_str = "; ".join(f"{k}: {', '.join(v)}" for k, v in cyp_modes.items())
        qc_flags.append(f"CELL-LINE MISMATCH (CYP2D6i Swap): Mixed cell backgrounds -> {cyp_mode_str}.")

# Signature drift check between Cell 3 CYP2D6i and Cell 6 matching inhibitor
if (not sig_df_r.empty and "CYP2D6i" in sig_df_r.columns
        and not cyp_agents_r.empty and CYP2D6_INHIBITOR in set(cyp_agents_r["CYP2D6i"])):
    row6 = cyp_agents_r[cyp_agents_r["CYP2D6i"] == CYP2D6_INHIBITOR].iloc[0]
    drifts = []
    for g in ["CDC42", "SLC1A3", "GLUL", "AKT3"]:
        if g in sig_df_r.index and g in row6.index and pd.notna(row6[g]):
            v3, v6 = float(sig_df_r.loc[g, "CYP2D6i"]), float(row6[g])
            if abs(v3 - v6) > 0.5:
                drifts.append(f"{g} (Cell3:{v3:+.2f} vs Cell6:{v6:+.2f})")
    if drifts:
        qc_flags.append(
            f"SIGNATURE SAMPLING DRIFT ({CYP2D6_INHIBITOR}): Unordered API pagination shifted consensus values -> "
            + ", ".join(drifts) + ". Fix: sort signatures deterministically by ID or increase MAX_SIGS_PER_COMPOUND."
        )

if not sig_meta_r.empty:
    zero_ag = sig_meta_r[sig_meta_r["n_used"] == 0]["agent"].tolist()
    if zero_ag:
        qc_flags.append(f"MISSING CGR ARMS: {', '.join(zero_ag)} returned 0 LINCS L1000 signatures and are absent from composites.")
    one_ag = sig_meta_r[sig_meta_r["n_used"] == 1]["agent"].tolist()
    if one_ag:
        qc_flags.append(f"SINGLE-SIGNATURE HIGH VARIANCE: {', '.join(one_ag)} based on n=1 L1000 signature.")

for dis, d_df in all_loaded_degs.items():
    ov = len(set(d_df["symbol"]) & set(covered_mod))
    if ov < 10:
        qc_flags.append(f"UNDERPOWERED DEG OVERLAP: {dis} shares only {ov} genes with L1000 synaptic modules (min_n=10 required).")

if qc_flags:
    for idx, msg in enumerate(qc_flags, 1):
        lines.append(f"  {idx}. {msg}")
else:
    lines.append("  [OK] No methodological or QC flags triggered.")

# -------------------------------------------------------------------------
# 10. ARTIFACT INVENTORY, ZIP BUNDLE & DOWNLOAD
# -------------------------------------------------------------------------
lines.append(_hr("10. GENERATED OUTPUT ARTIFACTS"))
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

for p in sorted(Path("results").glob("*")):
    if p.is_file():
        lines.append(f"  - {p.name:52s} ({p.stat().st_size:>10,} bytes)")
final_text = "\n".join(lines) + "\n"
summary_path.write_text(final_text, encoding="utf-8")

zip_path = Path("cgr_transcriptomic_benchmark_all_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path("results").glob("*")):
        if p.is_file():
            zf.write(p, arcname=p.name)

print(f"[OK] Wrote master summary ({len(final_text):,} chars, {len(lines)} lines) -> {summary_path}")
print(f"[OK] Bundled all results -> {zip_path} ({zip_path.stat().st_size:,} bytes)\n")
print(final_text[:2800] + ("\n... [truncated preview; open downloaded .txt for full report] ..." if len(final_text) > 2800 else ""))

from google.colab import files
files.download(str(summary_path))

[OK] Wrote master summary (36,636 chars, 252 lines) -> results/cgr_transcriptomic_benchmark_master_summary.txt
[OK] Bundled all results -> cgr_transcriptomic_benchmark_all_results.zip (1,075,312 bytes)

CGR TRANSCRIPTOMIC BENCHMARK — MASTER DETAILED ANALYSIS SUMMARY
Generated (UTC)         : 2026-11-25 17:26:49 UTC
Primary CYP2D6 Inhibitor: bupropion
Prefer Neural Cells     : False
Include Adhesion/PSD    : False
Max Sigs per Compound   : 60
Total Anchor Genes      : 121
Target Module Genes     : 104

1. SYNAPTIC MODULE & LINCS L1000 GENE COVERAGE
Consensus L1000 Matrix  : 12,327 total genes x 9 columns
Target Module Coverage  : 93 / 104 genes (89.4%)
Missing Module Genes (11): ACTG1, CALM1, CALM2, PLA2G4B, PLA2G4D, PLA2G4E, PLA2G4F, RPTOR, SHANK3, SLC17A8, SLC38A3
  - LTP_init      :  67 defined |  58 covered in L1000 (86.6%)
  - TrkB_mTOR     :  45 defined |  41 covered in L1000 (91.1%)
  - Actin_spine   :  27 defined |  25 covered in L1000 (92.6%)

2. LINCS L1000 SIGNATURE INVENTORY

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!cp "/content/results" "/content/drive/MyDrive/Dr Uccello/00_Studies/102_synapse" -r

# The End